# Converting $\LaTeX$ to <span style="font-variant:small-caps;">Html</span>

The purpose of the following exercise is to implement a translator from [$\LaTeX$](http://www.latex-project.org) to 
[MathML](https://www.tutorialspoint.com/mathml/index.htm).  $\LaTeX$ is a document markup language
that is especially well suited to present text that contains mathematical formulas.  MathML is the part of <span style="font-variant:small-caps;">Html</span> that deals with the representation of mathematical formulas.  As $\LaTeX$ provides a very rich
document markup language and we can only afford to spend a few hours on this exercise, we confine
ourselves to a small subset of $\LaTeX$.  The file `example.tex` contains some $\LaTeX$.  The goal of this exercise is to implement a translator that is able to transform this file into MathML.

The translator is implemented using nothing but Python's built-in module `re`.  In contrast to the
scanner developed in the notebook `01-RegExp-Scanner.ipynb`, this scanner needs two additional
ingredients:
1. It uses **scanner states**, because the rules that apply inside a mathematical formula are
   different from the rules that apply in ordinary text.
2. It uses a **stack**, because once we encounter a closing brace `}` we have to know which
   construct is closed by this brace.  As these constructs can be nested, a single variable is not
   enough to store this information.

We start with reading the file. 

In [ ]:
with open('example.tex') as f:
    data = f.read()

Now data contains the text that is stored in this file.

In [ ]:
print(data)

Let us look at the output file `example.pdf` that would be produced if we would run $\LaTeX$ on this file. 
If you are not running <span style="font-variant:small-caps;">MacOS</span> you have to replace the command
`open` with the name of an executable that can open a `.pdf`-file.

In [ ]:
!open example.pdf

Next, we open the file `example.html`.  The scanner we are going to implement has to write its output into this file.

In [ ]:
outfile = open('example.html', 'w')

<hr style="height:4px;background-color:blue">
Below are some predefined functions that you can use to create the <span style="font-variant:small-caps;">Html</span> file.
<hr style="height:4px;background-color:blue">

The function `start_html` writes the header of the <span style="font-variant:small-caps;">Html</span> file
and the opening `<body>` tag to the file opened above.  The `<script>` tag loads
[MathJax](https://www.mathjax.org), which is the *JavaScript* library that renders the
<span style="font-variant:small-caps;">MathML</span> that we are going to produce.

In [ ]:
def start_html():
    outfile.write('<!DOCTYPE html>\n')
    outfile.write('<html lang="en">\n')
    outfile.write('<head>\n')
    outfile.write('<meta charset="utf-8">\n')
    outfile.write('<title>Converted LaTeX Document</title>\n')
    outfile.write('<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js" async>\n')
    outfile.write('</script>\n')
    outfile.write('</head>\n')
    outfile.write('<body>\n\n')

The function `end_html` writes the closing `</body>` and `</html>` tags.

In [ ]:
def end_html():
    outfile.write('</body>\n')
    outfile.write('</html>\n')

The function `start_math_block` starts a math block.  This is useful for formulas enclosed in `$$`.  This type of formulas is displayed in a line by itself.

In [ ]:
def start_math_block():
    outfile.write('<math xmlns="http://www.w3.org/1998/Math/MathML" display="block">\n')

The function `start_math_inline` starts an <em style="color:blue">inline formula</em>, i.e. a formula enclosed in `$`.  Formulas of this type are part of the surrounding text.

In [ ]:
def start_math_inline():
    outfile.write('<math xmlns="http://www.w3.org/1998/Math/MathML" display="inline">\n')

The function `end_math` ends a math block.

In [ ]:
def end_math():
    outfile.write('</math>\n')

The functions `start_sum` and `end_sum` write code to display formulas involving sums.  For example, to display  the expression
$$ \sum\limits_{i=1}^n i^2 $$
we can use the following MathML:
```
<munderover>
<mo>&sum;</mo>
<mrow>
<mi>i</mi>
<mo>=</mo>
<mn>1</mn>
</mrow>
<mrow>
<mi>n</mi>
</mrow>
</munderover>
<msup>
<mi>i</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>

```

In [ ]:
def start_sum():
    outfile.write('<munderover>\n')
    outfile.write('<mo>&sum;</mo>\n')

def end_sum():
    outfile.write('</munderover>\n')

The functions `start_sqrt` and `end_sqrt` write code to display formulas involving square roots.  For example, to display  the expression
$$ \sqrt{a^2 + b^2} $$
we can use the following MathML:
```
<msqrt>
<mrow>
<msup>
<mi>a</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
<mo>+</mo>
<msup>
<mi>b</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
</mrow>
</msqrt>
```

In [ ]:
def start_sqrt():
    outfile.write('<msqrt>\n')

def end_sqrt():
    outfile.write('</msqrt>\n')

In order to write exponents we have to use the tag `<msup>`.  For example, the expression $a^2$ 
is equivalent to the following markup:
```
<msup>
<mi>a</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
```
Note that the exponent is enclosed in `<mrow>` `</mrow>` tags.

In [ ]:
def start_super():
    outfile.write('<msup>\n')

def end_super():
    outfile.write('</msup>\n')

In order to write fractions we have to use the tag `<mfrac>`.  For example, the expression $\frac{1}{6}$ 
is equivalent to the following markup:
```
<mfrac>
<mrow>
<mn>1</mn>
</mrow>
<mrow>
<mn>6</mn>
</mrow>
</mfrac>
```
Note that both nominator and denominator are enclosed in `<mrow>` `</mrow>` tags.

In [ ]:
def start_fraction():
    outfile.write('<mfrac>\n')

def end_fraction():
    outfile.write('</mfrac>\n')

Arguments of functions like the square root or exponents have to be enclosed in pairs of `<mrow>` and `</mrow>` tags.  

In [ ]:
def start_row():
    outfile.write('<mrow>\n')

def end_row():
    outfile.write('</mrow>\n')

Variable names should be enclosed in pairs of `<mi>` and `</mi>` tags.  For example, the variable $x$ is displayed by the following MathML:
```
<mi>x</mi>
```

In [ ]:
def write_var(v):
    outfile.write('<mi>' + v + '</mi>\n')

Numbers should be enclosed in pairs of `<mn>` and `</mn>` tags.  For example, the number $6$ is displayed by the following MathML:
```
<mn>6</mn>
```

In [ ]:
def write_number(n):
    outfile.write('<mn>' + n + '</mn>\n')

The symbol $\cdot$ is created by the following MathML:
```
<mo>&sdot;</mo>
```

In [ ]:
def write_times():
    outfile.write('<mo>&sdot;</mo>\n')

Mathematical operators should be enclosed in pairs of `<mo>` and `</mo>` tags.  For example, the operator $+$ is displayed by the following MathML:
```
<mo>+</mo>
```

In [ ]:
def write_operator(op):
    outfile.write('<mo>' + op + '</mo>\n')

The symbol $\pi$ is created by the following MathML:
```
<mn>&pi;</mn>
```

In [ ]:
def write_pi():
    outfile.write('<mn>&pi;</mn>\n')

The symbol $\leq$ is created by the following MathML:
```
<mo>&le;</mo>
```

In [ ]:
def write_leq():
    outfile.write('<mo>&le;</mo>\n')

The symbol $\geq$ is created by the following MathML:
```
<mo>&ge;</mo>
```

In [ ]:
def write_geq():
    outfile.write('<mo>&ge;</mo>\n')

The function `write_any` writes a single character unadorned to the output file.

In [ ]:
def write_any(char):
    outfile.write(char)

<hr style="height:4px;background-color:blue">
The scanner starts here.
<hr style="height:4px;background-color:blue">

## Imports

The only module that we need is the module `re`.

In [ ]:
import re

## Scanner States

Our scanner has to behave differently in different parts of the input file.  Therefore it is
organized as a [finite state machine](https://en.wikipedia.org/wiki/Finite-state_machine) that has
the following four states:
- `PREAMBLE` is the state the scanner starts in.  Everything that occurs before the string
  `\begin{document}` belongs to the preamble of the $\LaTeX$ document and is discarded.
- `TEXT` is the state that the scanner is in when it reads ordinary text, i.e. text that is *not*
  part of a mathematical formula.  This text is echoed unchanged to the output file.
- `FORMULA` is the state that the scanner is in once it has read either a `$` or a `$$`.  In this
  state the input has to be translated into <span style="font-variant:small-caps;">MathML</span>.
- `DONE` is the state that is reached once the string `\end{document}` has been read.  In this
  state the scanner stops.

For every state we need a separate token specification.  Note that in the state `DONE` the scanner
does not read anything anymore, so no token specification is needed for this state.

## Specifying the Tokens

The dictionary `token_specifications` maps every state to the list of token specifications that is
used in this state.  As in the notebook `01-RegExp-Scanner.ipynb`, every token specification is a
pair: The first component is the name of the token, while the second component is a regular
expression describing the strings that are recognized as tokens of this type.

Since Python's `re` module always chooses the **first** alternative that matches at the current
position, the order of these pairs is important:
- `VARIABLE_HAT` has to precede `VARIABLE`, for otherwise the string `i^{` would be scanned as the
  variable `i`.
- `CB_HAT_OB` has to precede `CB_OB`, which in turn has to precede `CB`, for otherwise the string
  `}^{` would be scanned as a single closing brace.
- The last specification of the states `PREAMBLE`, `TEXT`, and `FORMULA` matches an arbitrary
  character.  Hence, in these states a match is always found and we never have to check whether
  `match` has returned `None`.  In the state `FORMULA` this catch-all token is called `MISMATCH`,
  because any character that is not covered by one of the preceding regular expressions is an
  error, while in the states `PREAMBLE` and `TEXT` arbitrary characters are perfectly acceptable.

In [ ]:
token_specifications = {
    'PREAMBLE': [
        ('HEAD',           r'\\documentclass\{article\}'),
        ('BEGIN_DOCUMENT', r'\\begin\{document\}'),
        ('SKIP',           r'[\s\S]'),                    # discard everything else
    ],
    'TEXT': [
        ('END_DOCUMENT',   r'\\end\{document\}'),
        ('DOLLAR_DOLLAR',  r'\$\$'),
        ('DOLLAR',         r'\$'),
        ('ANY',            r'[\s\S]'),                    # echo everything else
    ],
    'FORMULA': [
        ('DOLLAR_DOLLAR',  r'\$\$'),
        ('DOLLAR',         r'\$'),
        ('SUM_LIMITS',     r'\\sum\\limits\s*_\s*\{'),
        ('SQRT',           r'\\sqrt\s*\{'),
        ('FRAC',           r'\\frac\s*\{'),
        ('CDOT',           r'\\cdot'),
        ('LEQ',            r'\\leq'),
        ('GEQ',            r'\\geq'),
        ('PI',             r'\\pi'),
        ('VARIABLE_HAT',   r'[a-zA-Z]+\s*\^\s*\{'),
        ('NUMBER_HAT',     r'(?:0|[1-9][0-9]*)\s*\^\s*\{'),
        ('CB_HAT_OB',      r'\}\s*\^\s*\{'),
        ('CB_OB',          r'\}\s*\{'),
        ('CB',             r'\}'),
        ('NUMBER',         r'0|[1-9][0-9]*'),
        ('VARIABLE',       r'[a-zA-Z]+'),
        ('OPERATOR',       r'[-+*/=<>.,()]'),
        ('WS',             r'\s+'),                       # whitespace is not significant
        ('MISMATCH',       r'[\S]'),                    # anything else is an error
    ],
}

## Building the Master Patterns

For every state we combine the regular expressions of the corresponding token specification into a
single *master pattern*.  As before, every regular expression is wrapped into a *named group* of
the form
```
(?P<NAME>regexp)
```
and these named groups are joined using the alternative operator `|`.  Note that the names of the
groups only have to be unique **within** a single master pattern.  Therefore the token
`DOLLAR_DOLLAR` can occur both in the state `TEXT` and in the state `FORMULA`.

In contrast to the notebook `01-RegExp-Scanner.ipynb` we compile the master patterns via
`re.compile`.  The reason is that we can not use the function `re.finditer` here: as the scanner
switches between different patterns, we have to control the position where the next match has to
start ourselves.  This is done via the method `match` of a compiled regular expression, which takes
the position as its second argument.

In [ ]:
master_patterns = { state: re.compile('|'.join(f'(?P<{name}>{regexp})' for name, regexp in spec))
                    for state, spec in token_specifications.items()
                  }

Let us take a look at the master pattern that is used inside a formula:

In [ ]:
master_patterns['FORMULA'].pattern

## Processing the Tokens of the State `PREAMBLE`

The function `process_preamble` takes a token that has been found in the state `PREAMBLE` and
returns the state that the scanner has to switch to.  Everything that is read in this state is
discarded.  Once the string `\begin{document}` is found, the header of the
<span style="font-variant:small-caps;">Html</span> file is written and the scanner switches to the
state `TEXT`.

In [ ]:
def process_preamble(token, value):
    if token == 'BEGIN_DOCUMENT':
        start_html()
        return 'TEXT'
    return 'PREAMBLE'

## Processing the Tokens of the State `TEXT`

In the state `TEXT` the scanner echoes its input unchanged.  There are three exceptions:
- The string `$$` starts a formula that is displayed in a line of its own.
- The string `$` starts an inline formula.
- The string `\end{document}` ends the document.

In [ ]:
def process_text(token, value):
    if token == 'END_DOCUMENT':
        end_html()
        return 'DONE'
    if token == 'DOLLAR_DOLLAR':
        start_math_block()
        return 'FORMULA'
    if token == 'DOLLAR':
        start_math_inline()
        return 'FORMULA'
    write_any(value)   # token == 'ANY'
    return 'TEXT'

## Processing the Tokens of the State `FORMULA`

This is the interesting part.  The problem that we have to solve is the following: Once we
encounter a closing brace `}`, we have to know whether this brace closes the argument of a square
root, the lower or upper limit of a sum, the numerator or the denominator of a fraction, or an
exponent.  As these constructs can be nested, a single variable is not sufficient to store this
information.  Therefore, the function `process_formula` takes a list `stack` as its third argument.
Every time a construct that has to be closed by a brace is opened, we push a corresponding marker
onto this stack.  The markers that we use are the following:
- `'SQRT'`     is pushed when the string `\sqrt{` is read,
- `'FRAC_NUM'` is pushed when the string `\frac{` is read, i.e. when the numerator starts,
- `'FRAC_DEN'` is pushed when the string `}{` is read, i.e. when the denominator starts,
- `'SUM_LOWER'` is pushed when the string `\sum\limits_{` is read,
- `'SUM_UPPER'` is pushed when the string `}^{` is read that separates the limits of a sum,
- `'SUPER'`    is pushed when a string like `i^{` is read.

When a closing brace is found, the marker on top of the stack is popped and tells us which
<span style="font-variant:small-caps;">MathML</span> tags have to be written.  Note that the
markers `'FRAC_NUM'` and `'SUM_LOWER'` must not be closed by a simple `}`: the numerator of a
fraction is followed by its denominator, while the lower limit of a sum is followed by its upper
limit.

If the stack is not empty when a formula ends, the input contains an unbalanced opening brace and
an exception is raised.

In [ ]:
def process_formula(token, value, stack):
    if token in ('DOLLAR_DOLLAR', 'DOLLAR'):
        if stack:
            raise SyntaxError(f'formula ended while {stack[-1]} was still open')
        end_math()
        return 'TEXT'
    if token == 'SUM_LIMITS':
        start_sum()
        start_row()
        stack.append('SUM_LOWER')
    elif token == 'SQRT':
        start_sqrt()
        start_row()
        stack.append('SQRT')
    elif token == 'FRAC':
        start_fraction()
        start_row()
        stack.append('FRAC_NUM')
    elif token in ('VARIABLE_HAT', 'NUMBER_HAT'):
        base = value[:value.index('^')].strip()   # chop off the trailing '^{'
        start_super()
        if token == 'VARIABLE_HAT':
            write_var(base)
        else:
            write_number(base)
        start_row()
        stack.append('SUPER')
    elif token == 'CB_HAT_OB':                    # the string '}^{'
        if not stack or stack[-1] != 'SUM_LOWER':
            raise SyntaxError('"}^{" is only allowed to separate the limits of a sum')
        stack.pop()
        end_row()
        start_row()
        stack.append('SUM_UPPER')
    elif token == 'CB_OB':                        # the string '}{'
        if not stack or stack[-1] != 'FRAC_NUM':
            raise SyntaxError('"}{" is only allowed to separate numerator and denominator')
        stack.pop()
        end_row()
        start_row()
        stack.append('FRAC_DEN')
    elif token == 'CB':                           # the string '}'
        if not stack:
            raise SyntaxError('closing brace "}" without a matching opening brace')
        top = stack.pop()
        if top == 'SQRT':
            end_row()
            end_sqrt()
        elif top == 'FRAC_DEN':
            end_row()
            end_fraction()
        elif top == 'SUM_UPPER':
            end_row()
            end_sum()
        elif top == 'SUPER':
            end_row()
            end_super()
        else:
            raise SyntaxError(f'"}}" can not close {top}')
    elif token == 'NUMBER':
        write_number(value)
    elif token == 'VARIABLE':
        write_var(value)
    elif token == 'PI':
        write_pi()
    elif token == 'CDOT':
        write_times()
    elif token == 'LEQ':
        write_leq()
    elif token == 'GEQ':
        write_geq()
    elif token == 'OPERATOR':
        write_operator(value)
    elif token == 'WS':
        pass
    elif token == 'MISMATCH':
        raise SyntaxError(f'unexpected character {value!r} inside a formula')
    return 'FORMULA'

## The Scanner

The function `scan` drives the whole process.  It maintains three variables:
- `state` is the current state of the scanner.  It starts as `'PREAMBLE'`.
- `stack` stores the markers that are needed to interpret a closing brace.
- `pos`   is the position where the next token starts.

In every iteration of the loop the master pattern of the current state is matched against the input
`text` at the position `pos`.  Note that the method `match` only tries to find a match that starts
exactly at the given position, which is precisely what a scanner needs.  We then ask the match
object `mo` for the name of the group that is responsible for the match and dispatch to the
function that processes the tokens of the current state.  This function returns the state that the
scanner is in after the token has been processed.

In [ ]:
def scan(text):
    state = 'PREAMBLE'
    stack = []
    pos   = 0
    while state != 'DONE' and pos < len(text):
        mo    = master_patterns[state].match(text, pos)
        token = mo.lastgroup
        value = mo.group()
        pos   = mo.end()
        if state == 'PREAMBLE':
            state = process_preamble(token, value)
        elif state == 'TEXT':
            state = process_text(token, value)
        else:
            state = process_formula(token, value, stack)
    if state != 'DONE':
        raise SyntaxError(r'the input did not contain the string "\end{document}"')

## Running the Translator

Now we can translate the file `example.tex` that we have read into the variable `data`.

In [ ]:
scan(data)

In [ ]:
outfile.close()

Let us inspect the <span style="font-variant:small-caps;">Html</span> file that has been produced.

In [ ]:
print(open('example.html').read())

Finally, we open the file in a browser.  Again, if you are not running
<span style="font-variant:small-caps;">MacOS</span> you have to replace the command `open`.

In [ ]:
!open example.html

## What Happens if the Input is Malformed?

The functions defined above write their output to the global variable `outfile`.  Since this file
has already been closed, we redirect the output into a `StringIO` object, which behaves like a file
but stores everything in memory.  This way we can experiment without overwriting `example.html`.

In [ ]:
import io

outfile = io.StringIO()

The formula below contains an opening brace that is never closed.

In [ ]:
broken = r'''\documentclass{article}
\begin{document}
$$ \frac{1}{6 $$
\end{document}
'''
try:
    scan(broken)
except SyntaxError as error:
    print(error)

The formula below contains a character that is not supported inside a formula.

In [ ]:
outfile = io.StringIO()
broken = r'''\documentclass{article}
\begin{document}
$$ a \wedge b $$
\end{document}
'''
try:
    scan(broken)
except SyntaxError as error:
    print(error)

## Exercises

The translator implemented above supports only a small subset of $\LaTeX$.  Try to extend it:
1. Support the commands `\alpha`, `\beta`, and `\gamma`, which have to be translated into the
   <span style="font-variant:small-caps;">Html</span> entities `&alpha;`, `&beta;`, and `&gamma;`.
2. Support subscripts, i.e. expressions of the form `x_{1}`.  In
   <span style="font-variant:small-caps;">MathML</span>, a subscript is written using the tag
   `<msub>`.
3. Support the command `\sqrt` with an optional argument, i.e. expressions of the form
   `\sqrt[3]{x}`, which denote the third root of $x$.  The corresponding
   <span style="font-variant:small-caps;">MathML</span> tag is `<mroot>`.
4. In the state `TEXT` the characters `<`, `>`, and `&` are echoed unchanged.  Strictly speaking,
   these characters have to be replaced by the entities `&lt;`, `&gt;`, and `&amp;`.  Add a token
   that takes care of this.
5. Ordinary text is currently not wrapped into paragraphs.  Add a token that recognizes an empty
   line and translates it into the tag `<p>`.